In [ ]:
# prompt: mount gdrive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Load dataset
file_path =  '/content/drive/MyDrive/Colab Notebooks/dataset_build_complete.csv'

df = pd.read_csv(file_path, encoding='latin-1')

In [ ]:
# prompt: Stop warning

import warnings
warnings.filterwarnings('ignore')

In [ ]:

def encode_features(df, user_input=None):
    """
    Encode categorical features using one-hot encoding
    """
    df_encoded = df.copy()

    df_encoded = df_encoded.dropna(subset=['transmission', 'body_type', 'engine_capacity', 'price', 'model_year']) # Drop nulls


    # Initialize OneHotEncoder
    categorical_features = ['transmission', 'body_type']
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

    # Fit and transform categorical features
    encoded_cats = encoder.fit_transform(df_encoded[categorical_features])

    # Get feature names after encoding
    feature_names = encoder.get_feature_names_out(categorical_features)

    # Create DataFrame with encoded categorical features
    encoded_cats_df = pd.DataFrame(
        encoded_cats,
        columns=feature_names,
        index=df_encoded.index
    )

    # Numerical features remain the same
    numerical_features = ['engine_capacity', 'price', 'car_age']

    # Combine numerical and encoded categorical features
    encoded_df = pd.DataFrame()

    # Add numerical features
    for feat in numerical_features:
        encoded_df[feat] = df_encoded[feat]

    # Add one-hot encoded features
    for col in feature_names:
        encoded_df[col] = encoded_cats_df[col]

    if user_input is not None:
        user_df = pd.DataFrame([user_input])

        # Transform user input categorical features
        user_encoded_cats = encoder.transform(user_df[categorical_features])
        user_encoded_cats_df = pd.DataFrame(
            user_encoded_cats,
            columns=feature_names
        )

        # Create user encoded DataFrame
        user_encoded = pd.DataFrame()

        # Add numerical features
        for feat in numerical_features:
            user_encoded[feat] = user_df[feat]

        # Add one-hot encoded features
        for col in feature_names:
            user_encoded[col] = user_encoded_cats_df[col]

        # Ensure columns are in the same order
        user_encoded = user_encoded[encoded_df.columns]
        return encoded_df, user_encoded

    return encoded_df

def calculate_recommendation_metrics(X, k=5):
    """
    Calculate recommendation metrics using cross-validation
    """
    X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
    knn = NearestNeighbors(n_neighbors=k, metric='euclidean')
    knn.fit(X_train)

    correct_recommendations = 0
    total_recommendations = 0
    similarity_scores = []
    relevant_positions = []
    recommended_items_set = set()
    precision_at_k_list = []
    recall_at_k_list = []

    for idx, test_instance in X_test.iterrows():
        test_instance_array = test_instance.values.reshape(1, -1)
        distances, indices = knn.kneighbors(test_instance_array)
        recommended_features = X_train.iloc[indices[0]]
        recommended_items_set.update(indices[0])

        relevant_items = 0
        position_sum = 0
        current_similarities = []

        total_relevant = len(X_train[np.mean(X_train == test_instance, axis=1) >= 0.95])

        for pos, (_, rec_features) in enumerate(recommended_features.iterrows(), 1):
            similarity = np.mean(test_instance == rec_features)
            current_similarities.append(similarity)
            similarity_scores.append(similarity)

            if similarity >= 0.75:
                relevant_items += 1
                position_sum += pos

            if similarity >= 0.7:
                correct_recommendations += 1

            total_recommendations += 1

        if k > 0:
            precision_at_k_list.append(relevant_items / k)
            recall_at_k_list.append(relevant_items / total_relevant if total_relevant > 0 else 0)

        if relevant_items > 0:
            relevant_positions.append(position_sum / relevant_items)

    precision_at_k = np.mean(precision_at_k_list) if precision_at_k_list else 0
    recall_at_k = np.mean(recall_at_k_list) if recall_at_k_list else 0

    metrics = {
        'accuracy': correct_recommendations / max(total_recommendations, 1),
        'mean_similarity': np.mean(similarity_scores) if similarity_scores else 0,
        'coverage': len(recommended_items_set) / len(X_train),
        'precision_at_k': precision_at_k,
        'recall_at_k': recall_at_k,
        'mean_relevant_position': np.mean(relevant_positions) if relevant_positions else 0,
        'f1_score': 2 * (precision_at_k * recall_at_k) / (precision_at_k + recall_at_k) if (precision_at_k + recall_at_k) > 0 else 0
    }

    return metrics

def recommend_cars(df, user_input):
    """
    Recommend cars based on user preferences
    """
    # Calculate car_age for the dataset
    df['car_age'] = 2024 - df['model_year']

    df_encoded, user_encoded = encode_features(df, user_input)

    # Scale numerical features
    scaler = StandardScaler()
    numerical_features = ['engine_capacity', 'price', 'car_age']
    df_encoded[numerical_features] = scaler.fit_transform(df_encoded[numerical_features])
    user_encoded[numerical_features] = scaler.transform(user_encoded[numerical_features])

    # Calculate recommendation metrics
    recommendation_metrics = calculate_recommendation_metrics(df_encoded)

    # Fit KNN for recommendations
    knn = NearestNeighbors(n_neighbors=10, metric='euclidean')
    knn.fit(df_encoded)

    # Get recommendations
    distances, indices = knn.kneighbors(user_encoded)
    recommendations = df.iloc[indices[0]].copy()
    recommendations['similarity_score'] = 1 - (distances[0] / distances[0].max())

    return recommendations, recommendation_metrics



In [ ]:
# Example usage
user_input = {
    'body_type': 'Saloon',
    'transmission': 'Automatic',
    'engine_capacity': 1500,
    'price': 3050000,
    'car_age': 4
}

# Get recommendations and metrics
recommendations, metrics = recommend_cars(df, user_input)

# Print metrics
if metrics is not None:
    print("\nRecommendation System Performance Metrics:")
    print("-" * 40)
    print(f"Accuracy: {metrics['accuracy']:.3f}")
    print(f"Mean Similarity Score: {metrics['mean_similarity']:.3f}")
    print(f"Coverage: {metrics['coverage']:.3f}")
    print(f"Precision@K: {metrics['precision_at_k']:.3f}")
    print(f"Recall@K: {metrics['recall_at_k']:.3f}")
    print(f"Mean Relevant Position: {metrics['mean_relevant_position']:.2f}")
    print(f"F1 Score: {metrics['f1_score']:.3f}")

# Print recommendations
if not recommendations.empty:
    print(recommendations[[
        'brand', 'car_model', 'price', 'model_year', 'transmission', 'body_type', 'fuel_type',
        'engine_capacity', 'travels', 'similarity_score'
    ]].sort_values('similarity_score', ascending=False).to_string())


Recommendation System Performance Metrics:
----------------------------------------
Accuracy: 0.998
Mean Similarity Score: 0.891
Coverage: 0.677
Precision@K: 0.998
Recall@K: 0.928
Mean Relevant Position: 3.00
F1 Score: 0.962
       brand car_model      price  model_year transmission  body_type fuel_type  engine_capacity  travels  similarity_score
129   Toyota      Axio  3340000.0        2020    Automatic     Saloon    Octane             1500  48000.0          0.988992
128   Toyota      Axio  3340000.0        2020    Automatic     Saloon       CNG             1500  48000.0          0.988992
219   Toyota    Allion  3450000.0        2020    Automatic     Saloon    Octane             1500   1780.0          0.984816
218   Toyota    Allion  3450000.0        2020    Automatic     Saloon    Hybrid             1500   1780.0          0.984816
1095  Toyota      Axio  2275032.0        2014    Automatic  Hatchback    Octane             1500  82111.0          0.967564
1094  Toyota      Axio  227503